In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

In [ ]:


# 1. Define the directory path
folder_path = Path("../dataset/dsfsi-anv/anv")

# 2. Get all CSV files in the folder (use rglob("*.csv") to include subfolders)
csv_files = list(folder_path.glob("*.csv"))

In [ ]:
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df = df.astype({"language":'category',"split":'category','audio_id':'string','recorder_uuid':'string'},)
df = df.astype({'type':'category','domain':'category','topic':'category',})
df = df.astype({'scenario':'string','transcript':'string','document_id':'string','source_document':'category'})
df = df.astype({'microphone_device_id':'string','microphone_label':'category'})

In [ ]:
df.info(show_counts=True)

In [ ]:
df.isnull().sum()

In [ ]:
drop_coloumns = ['system_file_name','file_name','full_path']
display(df.drop(columns=drop_coloumns).describe(include='all'))
#drop_coloums = ['langauge','system_file_name','file_name','full_path']

In [ ]:
# Use display() combined with Markdown()
display(Markdown(f"## Now viewing: **Language**"))

plt.figure(figsize=(6, 4))
df['language'].value_counts().plot(kind="bar")
plt.title(f"Value Counts for Column: language")
plt.xlabel("Language")
plt.ylabel("Count")
plt.show()

In [ ]:
languages = df['language'].unique()
languages

In [ ]:
# lang_dict = {
#     "isiNdebele": "NBL",
#     "isiXhosa": "XHO",
#     "isiZulu": "ZUL",
#     "Sesotho": "SOT",
#     "Setswana": "TSN",
#     "Tshivenda": "VEN",
#     "Xitsonga": "TSO",
# }

lang_dict = {
    "Setswana": "TSN",
    "Tshivenda": "VEN",
}

In [ ]:
N = 2 #number of languages to display

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]
    display_coloumns = ['split','type','domain','microphone_label', 'source_document']
    for col in df_filtered[display_coloumns].columns:
        # Use display() combined with Markdown()
        display(Markdown(f"## Now viewing {language} ({code}): **{col}**"))

        plt.figure(figsize=(6, 4))
        df_filtered[col].value_counts().plot(kind="bar")
        plt.title(f"Value Counts for {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]
    display_coloumns = ['duration','size_bytes','signal_to_noise_ratio','audio_num_samples']
    for col in df_filtered[display_coloumns].columns:

        display(Markdown(f"## Now viewing: **{col}**"))

        # Pass the data array/series directly
        plt.hist(df_filtered[col], bins=15, color="skyblue", edgecolor="black")

        # Adding labels
        plt.title(f"Distribution of {col}")
        plt.xlabel(f"{col} Group")
        plt.ylabel("Count")

        # Display the plot
        plt.show()

        display(df_filtered[col].describe())

# Transcript Analysis

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]

    display(Markdown(f"## Now viewing: **Top words in {language} ({code})**"))

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_colwidth", None)

    for category in sorted(df_filtered['type'].dropna().unique()):
        subset = df_filtered[df_filtered['type'] == category][['type', 'transcript']].head(10)
        display(Markdown(f"### Type: {category}"))
        display(subset)

    #count none or empty transcripts
    empty_transcripts = df_filtered['transcript'].isnull().sum() + (df_filtered['transcript'].str.strip() == '').sum()
    display(Markdown(f"### Number of empty or null transcripts: {empty_transcripts}"))


    df_filtered['transcript_word_count'] = df_filtered['transcript'].apply(lambda x: len(str(x).split()))
    plt.hist(df_filtered['transcript_word_count'], bins=20, color="lightgreen", edgecolor="black")
    plt.title("Distribution of Transcript Word Counts")
    plt.xlabel("Word Count")
    plt.ylabel("Frequency")
    plt.show()


    display(Markdown(f"## Now viewing: **Top words in transcript**"))
    from collections import Counter
    all_words = ' '.join(df_filtered['transcript'].dropna()).lower().split()
    word_counts = Counter(all_words)
    top_words = word_counts.most_common(20)
    display(Markdown(f"### Top Words in Transcript"))
    for i, (word, count) in enumerate(top_words, start=1):
        display(Markdown(f"{i}. **{word}**: {count}"))

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]

    df_filtered = df_filtered.copy()
    df_filtered['transcript_char_count'] = df_filtered['transcript'].apply(lambda x: len(str(x)))

    plt.hist(df_filtered['transcript_char_count'], bins=20, color="lightgreen", edgecolor="black")
    plt.title(f"Distribution of Transcript Character Counts for {language} ({code})")
    plt.xlabel("Character Count")
    plt.ylabel("Frequency")
    plt.show()

    display(Markdown(f"## Now viewing: **Top characters in transcript for {language} ({code})**"))
    from collections import Counter
    all_chars = ''.join(df_filtered['transcript'].dropna()).lower()
    filtered_chars = [c for c in all_chars if not c.isspace()]
    char_counts = Counter(filtered_chars)
    top_chars = char_counts.most_common(20)

    number_of_unique_chars = len(char_counts)
    display(Markdown(f"### Number of Unique Characters in Transcript: {number_of_unique_chars}"))


    display(Markdown(f"### Top 20 Characters in Transcript"))
    for i, (char, count) in enumerate(top_chars, start=1):
        display(Markdown(f"{i}. **{char}**: {count}"))